In [1]:
%pip install -q --upgrade pip

# Install a ragas version that supports langchain-core >=0.3,.
# plus a compatible langchain-community pin to avoid the ChatVertexAI import error.
%pip install -q \
    "ragas>=0.2.15" \
    "langchain-google-genai>=2.0.0" \
    "langchain-community<0.4.2" \
    datasets pandas matplotlib seaborn

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from ragas import evaluate, EvaluationDataset 
from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
from ragas.dataset_schema import SingleTurnSample
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from ragas.embeddings import LangchainEmbeddingsWrapper

/tmp/ipykernel_1511/4279001877.py:7: DeprecationWarning: Importing SummarizationScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SummarizationScore
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_1511/4279001877.py:7: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_1511/4279001877.py:7: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Summarizat

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    response_mime_type="application/json",  # forces raw JSON, no fences
    google_api_key="",
)

embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",  # or "models/embedding-001"
        google_api_key="",
    )
)

/tmp/ipykernel_1511/4036962232.py:7: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(


In [4]:
ds = load_dataset("pameydorke/redred-gemma-4-E2B-it-lora-summaries", split="train")
df = ds.to_pandas()

# Show the columns and a sample row
print(df.columns.tolist())
df.head(2)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/394 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 93.5kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/22 [00:00<?, ? examples/s]

['user_input', 'reference', 'base_response', 'ft_response']


,user_input,reference,base_response,ft_response
0,Original Post: Help with Small living room Use...,The OP asked about general design suggestions ...,"The original poster, moving into a 1920s craft...",The user is asking for advice on how to arrang...
1,Original Post: What to do with this half a cyl...,The OP wanted advice on what to do with the wo...,The original poster asked for ideas on what to...,"A user on the ""designmyroom"" subreddit is aski..."


In [ ]:
def build_samples(row, response_col):
    """Create a SingleTurnSample for Ragas from a dataframe row."""
    return SingleTurnSample(
        user_input=row["user_input"],          # normalized Reddit thread text
        response=row[response_col],            # model summary (base or ft)
        reference=row["reference"],            # human‑made summary
        # SummarizationScore needs the original context to extract keyphrases
        reference_contexts=[row["user_input"]],
    )

# Build sample lists for the base model and the fine‑tuned model
base_samples = [build_samples(row, "base_response") for _, row in df.iterrows()]
ft_samples   = [build_samples(row, "ft_response")   for _, row in df.iterrows()]

base_dataset = EvaluationDataset(samples=base_samples)
ft_dataset = EvaluationDataset(samples=ft_samples)

In [ ]:
# Define the metrics we want to compute
metrics = [
    SummarizationScore(llm=llm, coeff=0.5),  # 0.5 balances QA vs. conciseness
    SemanticSimilarity(embeddings=embeddings),
    AnswerCorrectness(llm=llm, embeddings=embeddings),
]

# Evaluate the base model summaries
base_result = evaluate(
    dataset=base_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
)

# Evaluate the fine‑tuned model summaries
ft_result = evaluate(
    dataset=ft_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
)

# Convert results to dataframes for easy inspection
base_df = base_result.to_pandas()
ft_df   = ft_result.to_pandas()

print("Base model scores:")
print(base_df.mean(numeric_only=True))
print("\nFine‑tuned model scores:")
print(ft_df.mean(numeric_only=True))

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[17]: GoogleRateLimitError(Error calling model 'gemini-3.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 6.372874219s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaI

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[3]: ValidationError(1 validation error for ExtractedKeyphrases
  Invalid JSON: invalid escape at line 3 column 13 [type=json_invalid, input_value='{\n    "keyphrases": [\n...    "90\\\'s"\n    ]\n}', input_type=TextAccessor]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid)
ERROR:ragas.executor:Exception raised in Job[15]: GoogleRateLimitError(Error calling model 'gemini-3.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 10.872790446s.', 'status': 'RESOURCE_EXHAUSTED', 'd

KeyboardInterrupt: 

ERROR:ragas.executor:Exception raised in Job[44]: AssertionError(AnswerSimilarity must be set)
ERROR:ragas.executor:Exception raised in Job[30]: GoogleRateLimitError(Error calling model 'gemini-3.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 36.093772609s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [